# Experiments notebook

lets make ML for the Individual system 

# ML for a Single System

This notebook starts with an individual-system baseline using the collected host telemetry in `datasets/generated/system_metrics.csv`.

Next we will extend the same workflow to the distributed system view once the single-node baseline is in place.

In [16]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / 'datasets' / 'generated' / 'system_metrics.csv').exists():
        REPO_ROOT = candidate
        break
else:
    raise FileNotFoundError('Could not locate datasets/generated/system_metrics.csv')

DATA_PATH = REPO_ROOT / 'datasets' / 'generated' / 'system_metrics.csv'
df = pd.read_csv(DATA_PATH)
df.head()
# Create temporary labels

df["high_load"] = (
    (df["cpu"] > 85)
    | (df["memory"] > 90)
    | (df["disk"] > 95)
).astype(int)

print(df.shape)

print(df["high_load"].value_counts())

(66, 8)
high_load
0    49
1    17
Name: count, dtype: int64


## Clean and score the host telemetry

We convert the collected telemetry into a simple anomaly baseline. Rows that look unusual across CPU, memory, disk, and process load will be flagged for the first-pass model.

In [17]:
import pandas as pd
baseline_scores = pd.DataFrame(
    {
        'timestamp': df['timestamp'],
        'node_id': df['node_id'],
        'cpu': df['cpu'],
        'memory': df['memory'],
        'disk': df['disk'],
        'network': df['network'],
        'high_load': df['high_load'],
    }
)
print(df.shape)
print(df["high_load"].value_counts())
baseline_scores.sort_values(['high_load', 'cpu', 'memory'], ascending=[False, False, False]).head(10)

(66, 8)
high_load
0    49
1    17
Name: count, dtype: int64


,timestamp,node_id,cpu,memory,disk,network,high_load
0,2026-05-29T12:21:25.474687+00:00,test-node-1,92.0,88.0,91.0,12.00,1
8,2026-05-29T12:37:27.165563+00:00,LAPTOP-RSR05V3M,22.4,91.0,40.8,189918.70,1
7,2026-05-29T12:37:22.108437+00:00,LAPTOP-RSR05V3M,22.1,91.0,40.8,189412.77,1
9,2026-05-29T12:37:36.472365+00:00,LAPTOP-RSR05V3M,20.5,91.2,40.8,190517.53,1
10,2026-05-29T12:37:41.446610+00:00,LAPTOP-RSR05V3M,19.7,91.3,40.8,190639.21,1
6,2026-05-29T12:37:13.403453+00:00,LAPTOP-RSR05V3M,18.3,90.1,40.8,188124.15,1
11,2026-05-29T12:37:49.753886+00:00,LAPTOP-RSR05V3M,17.5,91.1,40.8,190668.61,1
12,2026-05-29T12:37:54.926185+00:00,LAPTOP-RSR05V3M,17.2,90.9,40.8,190767.04,1
2,2026-05-29T12:36:35.980162+00:00,LAPTOP-RSR05V3M,17.1,91.1,40.8,186713.57,1
13,2026-05-29T12:38:03.227385+00:00,LAPTOP-RSR05V3M,16.4,90.8,40.8,190792.79,1


In [18]:
numeric_columns = ['cpu', 'memory', 'disk', 'network']
df_model = df.copy()
df_model[numeric_columns] = df_model[numeric_columns].apply(pd.to_numeric, errors='coerce')
df_model = df_model.dropna(subset=numeric_columns).reset_index(drop=True)
df_model['process_load'] = df_model[numeric_columns].mean(axis=1)
df_model['high_load'] = ((df_model['cpu'] >= 85) | (df_model['memory'] >= 90) | (df_model['disk'] >= 95)).astype(int)

features = df_model[['cpu', 'memory', 'disk', 'network', 'process_load']]
labels = df_model['high_load']

X_train, X_test, y_train, y_test = train_test_split(features, labels, test_size=0.25, random_state=42, stratify=labels if labels.nunique() > 1 else None)
X_train.shape, X_test.shape

((49, 5), (17, 5))

## Train a baseline model

For the first pass, we use a lightweight anomaly detector on the single-system telemetry. This keeps the notebook fast and gives us a baseline before moving to the distributed case.

In [19]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest

model = Pipeline(
    steps=[
        ('scaler', StandardScaler()),
        ('detector', IsolationForest(n_estimators=200, contamination=0.1, random_state=42)),
    ]
)

# Fit the pipeline on training data
model.fit(X_train)

# Detector returns 1 for inliers and -1 for outliers; map to 0/1 labels
predictions = model.named_steps['detector'].predict(StandardScaler().fit_transform(X_test))
predictions = pd.Series(predictions).map({1: 0, -1: 1})

print('Confusion matrix:')
print(confusion_matrix(y_test, predictions))
print('\nClassification report:')
print(classification_report(y_test, predictions, zero_division=0))

Confusion matrix:
[[10  3]
 [ 0  4]]

Classification report:
              precision    recall  f1-score   support

           0       1.00      0.77      0.87        13
           1       0.57      1.00      0.73         4

    accuracy                           0.82        17
   macro avg       0.79      0.88      0.80        17
weighted avg       0.90      0.82      0.84        17



## Review model output

If the baseline is weak, the next step is to improve feature engineering before we move to distributed telemetry aggregation.

# Distributed System Next

Once the single-system baseline is stable, the distributed system version should aggregate node-level telemetry, compare enclaves, and train a cluster-level model over the combined snapshots.

In [20]:
# Save the trained pipeline to disk
from pathlib import Path
import joblib

models_dir = REPO_ROOT / 'ml_models'
models_dir.mkdir(parents=True, exist_ok=True)
model_path = models_dir / 'isolation_forest_v1.joblib'
joblib.dump(model, model_path)
print('Saved model to', model_path)

# Optionally, show file size
print('Model size (bytes):', model_path.stat().st_size)

Saved model to D:\projects\ML projects\Synapse Arc\ml_models\isolation_forest_v1.joblib
Model size (bytes): 536803
